# Nom du projet

**Objectif :** Décrire ici le problème métier/data science à résoudre et la variable cible à prédire.

> **Question principale :** Quelle question ce projet doit-il permettre de résoudre ?

### Résumé du projet
- **Type de problème :** Classification / Régression / Clustering / Série temporelle
- **Target :** `TARGET`
- **Métrique principale :** `F1` / `Accuracy` / `RMSE` / `MAE` / autre
- **Source des données :** `../data/`


## Import des librairies nécessaires

In [5]:
# Configuration générale
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Statistiques
from scipy.stats import ttest_ind, chi2_contingency

# Machine Learning
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# Modèles — à adapter au projet
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import joblib


## Configuration du projet

Centraliser ici les paramètres importants permet de réutiliser facilement le notebook.

**À modifier :**
- chemin du dataset
- nom de la target
- colonnes à supprimer
- type de problème
- `random_state`


In [6]:
DATA_PATH = "../data/telecom_customer_churn.csv"
TARGET = "TARGET"

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Colonnes à supprimer si elles ne sont pas pertinentes
DROP_COLUMNS = [
    # "id",
    # "name",
    # "url",
]

MODEL_PATH = "../backend/ml_models/model.pkl"


## Chargement du jeu de données

In [7]:
df = pd.read_csv(DATA_PATH).copy()

print(f"Shape : {df.shape}")
df.head()


Shape : (7043, 38)


,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


## Exploration du jeu de données — EDA

### Checklist de base

**Analyse de forme**
- Nombre de lignes :
- Nombre de colonnes :
- Variable cible :
- Variables numériques :
- Variables catégorielles :
- Variables identifiantes :
- Valeurs manquantes :

**Analyse de fond**
- Distribution de la target :
- Variables potentiellement importantes :
- Valeurs aberrantes :
- Déséquilibre éventuel des classes :
- Premières hypothèses :


### Analyse de forme

In [ ]:
print("Dimensions :", df.shape)
print("\nTypes de variables :")
print(df.dtypes.value_counts())

print("\nTypes détaillés :")
display(df.dtypes.to_frame("dtype"))

print("\nValeurs manquantes (%) :")
missing = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_%"))


### Aperçu statistique

In [ ]:
display(df.describe(include="all").T)


### Valeurs manquantes

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(df.isna(), cbar=False)
plt.title("Valeurs manquantes")
plt.show()


### Analyse de la target

In [ ]:
# Distribution de la variable cible
display(df[TARGET].value_counts(dropna=False))
display((df[TARGET].value_counts(normalize=True, dropna=False) * 100).round(2))


### Analyse des variables numériques

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_features = [col for col in numeric_cols if col != TARGET]

for col in numeric_features:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution de {col}")
    plt.show()


### Analyse des variables catégorielles

In [ ]:
categorical_features = df.select_dtypes(exclude=np.number).columns.tolist()

for col in categorical_features:
    print(f"\n{'-' * 60}\n{col}\n{'-' * 60}")
    display(df[col].value_counts(dropna=False).head(20))

    plt.figure(figsize=(8, 4))
    df[col].value_counts(dropna=False).head(15).plot(kind="bar")
    plt.title(f"Distribution de {col}")
    plt.xticks(rotation=45, ha="right")
    plt.show()


### Relation variables / target

In [ ]:
# Corrélations numériques
if numeric_features:
    plt.figure(figsize=(10, 7))
    corr_cols = numeric_features + [TARGET]
    sns.heatmap(df[corr_cols].corr(numeric_only=True), annot=True, cmap="Blues")
    plt.title("Matrice de corrélation")
    plt.show()


In [ ]:
# Comparaison des variables numériques selon la target
for col in numeric_features:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=TARGET, y=col)
    plt.title(f"{col} selon {TARGET}")
    plt.show()


## Test des hypothèses

Formuler ici des hypothèses **avant** les tests statistiques.

Exemples :
- H1 : `variable_A` est associée à la target.
- H2 : la moyenne de `variable_B` diffère selon les classes.
- H3 : `variable_C` n'apporte pas d'information utile.

> Les tests statistiques servent à compléter l'analyse exploratoire ; ils ne remplacent pas la validation du modèle.


In [ ]:
# Exemple : test du Chi² pour une variable catégorielle
def chi2_test(data, feature, target=TARGET, alpha=0.05):
    contingency = pd.crosstab(data[feature], data[target])
    stat, p_value, dof, expected = chi2_contingency(contingency)

    conclusion = "H0 rejetée" if p_value < alpha else "H0 non rejetée"

    return {
        "feature": feature,
        "test": "Chi²",
        "statistic": stat,
        "p_value": p_value,
        "conclusion": conclusion,
    }

# Exemple d'utilisation :
# chi2_test(df, "Sex")


In [ ]:
# Exemple : test de Student pour une variable numérique
def t_test(data, feature, target=TARGET, alpha=0.05):
    groups = data[target].dropna().unique()

    if len(groups) != 2:
        raise ValueError("Le t-test proposé ici nécessite exactement 2 classes.")

    g1 = data.loc[data[target] == groups[0], feature].dropna()
    g2 = data.loc[data[target] == groups[1], feature].dropna()

    stat, p_value = ttest_ind(g1, g2, equal_var=False)

    conclusion = "H0 rejetée" if p_value < alpha else "H0 non rejetée"

    return {
        "feature": feature,
        "test": "Welch t-test",
        "statistic": stat,
        "p_value": p_value,
        "conclusion": conclusion,
    }

# Exemple d'utilisation :
# t_test(df, "Age")


## Préparation des données

### Principes
1. Séparer les features (`X`) et la target (`y`).
2. Supprimer les colonnes inutiles ou identifiantes.
3. Faire le train/test split **avant** d'apprendre les transformations.
4. Utiliser un `Pipeline` pour éviter les fuites de données.
5. Adapter le preprocessing aux types de variables.


In [ ]:
# Suppression des colonnes inutiles
df_model = df.drop(columns=DROP_COLUMNS, errors="ignore").copy()

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y if y.nunique() == 2 else None,
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :")
display(y_train.value_counts(normalize=True))


In [ ]:
# Identification automatique des types de variables
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Variables numériques :", numeric_features)
print("Variables catégorielles :", categorical_features)


In [ ]:
# Preprocessing générique
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
)


## Modélisation

In [ ]:
# Modèle de référence
model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)

baseline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])

baseline.fit(X_train, y_train)


### Tester plusieurs modèles

Comparer plusieurs modèles permet d'éviter de retenir arbitrairement un modèle.

À adapter selon le problème :
- Classification : Logistic Regression, Random Forest, Gradient Boosting, XGBoost/LightGBM...
- Régression : Linear Regression, Random Forest Regressor, Gradient Boosting...


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE
    ),
}

results = []

for name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="f1_weighted",
    )

    results.append({
        "model": name,
        "cv_mean": scores.mean(),
        "cv_std": scores.std(),
    })

results_df = pd.DataFrame(results).sort_values("cv_mean", ascending=False)
display(results_df)


## Évaluation

In [ ]:
# Sélection du meilleur modèle
best_model_name = results_df.iloc[0]["model"]
best_estimator = models[best_model_name]

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_estimator),
])

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print(f"Modèle retenu : {best_model_name}")


In [ ]:
# Classification — à utiliser pour un problème de classification
print("Accuracy :", accuracy_score(y_test, y_pred))
print("F1 score :", f1_score(y_test, y_pred, average="weighted"))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion")
plt.show()


### Courbe d'apprentissage

In [ ]:
# Vérifier si le modèle souffre de sous-apprentissage ou de surapprentissage
train_sizes, train_scores, val_scores = learning_curve(
    final_model,
    X_train,
    y_train,
    cv=5,
    scoring="f1_weighted",
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1,
)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), label="Train")
plt.plot(train_sizes, val_scores.mean(axis=1), label="Validation")
plt.xlabel("Nombre d'observations")
plt.ylabel("F1 weighted")
plt.title("Courbe d'apprentissage")
plt.legend()
plt.show()


## Analyse des résultats

### Performance
- Score CV :
- Score test :
- Écart train / validation :
- Modèle retenu :

### Erreurs
- Faux positifs :
- Faux négatifs :
- Cas difficiles :

### Limites
- Taille du dataset :
- Variables manquantes :
- Biais potentiels :
- Risque de fuite de données :
- Généralisation :

### Pistes d'amélioration
1.
2.
3.


## Prédiction sur une nouvelle observation

In [ ]:
# Exemple générique : remplacer les valeurs par une observation réelle.
# Les noms de colonnes doivent correspondre à X.

new_data = pd.DataFrame([{
    # "feature_1": value,
    # "feature_2": value,
}])

# prediction = final_model.predict(new_data)
# prediction


## Sauvegarde du modèle

In [ ]:
# Le pipeline complet est sauvegardé :
# preprocessing + modèle
# Cela permet de réutiliser exactement les mêmes transformations en production.

MODEL_PATH = Path(MODEL_PATH)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, MODEL_PATH)

print(f"Modèle sauvegardé dans : {MODEL_PATH}")


## Conclusion

### Ce que nous avons appris
- 

### Réponse à la question initiale
- 

### Modèle final
- 

### Prochaine étape
- Déploiement / API / dashboard / amélioration du modèle


---

### Checklist finale

- [ ] Dataset chargé et vérifié
- [ ] Target correctement définie
- [ ] Valeurs manquantes analysées
- [ ] Variables explorées
- [ ] Hypothèses formulées
- [ ] Train/test split effectué avant le preprocessing
- [ ] Pipeline de preprocessing construit
- [ ] Plusieurs modèles comparés
- [ ] Validation croisée effectuée
- [ ] Métriques adaptées au problème
- [ ] Matrice de confusion / erreurs analysées
- [ ] Courbe d'apprentissage analysée
- [ ] Limites identifiées
- [ ] Modèle sauvegardé
- [ ] Notebook reproductible
